# PEYNNEY'S SOLUTION WITH AN EXTERNAL MAGNETIC FIELD IN ER

We refer the reader to the notebook EINSTEIN_Penney_Melvin for a complete description of the Penney solution. Note that to run this notebbok and in particular to verify the fild equations in ER, you need to have the python files expr_walkers.py and sage_fun.py stored in the same folder as this notebook. To reduce the computation time, one may also directly use the stored analytical expressions for the most computationally demanding quantities by placing the files ER_Penney_Melvin_dict.sobj and ER_Penney_Melvin_test_dict.sobj in the same directory and loading the corresponding expressions.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%display latex

In [3]:
version()

'SageMath version 10.1, Release Date: 2023-08-20'

'SageMath version 10.1, Release Date: 2023-08-20'

In [4]:
from sage.manifolds.operators import dalembertian
from sage.manifolds.operators import laplacian
from sage.manifolds.operators import grad
from sage.symbolic.operators import FDerivativeOperator
from sage.symbolic.expression_conversions import ExpressionTreeWalker
from expr_walkers import explore_expression, check_res_expr, decomp_exp
from expr_walkers import subs_func
import sage_func
import importlib
import sage.symbolic.expression_conversions as ec
importlib.reload(ec)

<module 'sage.symbolic.expression_conversions' from '/home/mwavasseur/Software/Sage/src/sage/symbolic/expression_conversions.py'>

In [5]:
functions_list=['Omega','Lambda','Av','Ev']
def subs_func(arg):
    subs_funcs = [(Omega, Omega_expr),(Lambda,Lambda_expr),(Av,Av_expr),(Ev,Ev_expr)] 
    if hasattr(arg, 'expr'):
        arg = arg.expr()
    if hasattr(arg, 'apply_map')*hasattr(arg, 'display'):
        for i, (old_func, new_func) in enumerate(subs_funcs):
            arg.apply_map(lambda f: f.substitute_function(old_func, new_func).simplify_full())
            print('{0} substituted'.format(functions_list[i])) 
    elif hasattr(arg, 'apply_map'):
        return arg.apply_map(lambda f: subs_func(f).simplify_full())
    else:
        for i, (old_func, new_func) in enumerate(subs_funcs):
            arg = arg.substitute_function(old_func, new_func).simplify_full()
            print('{0} substituted'.format(functions_list[i]))
    print('Full Substitution : Done')
    return arg

In [6]:
M = Manifold(4, 'M', structure='Lorentzian')
print(M)

4-dimensional Lorentzian manifold M


In [7]:
XY.<t,r,th,ph> = M.chart(r"t r:(0,+oo) th:(0,pi):\theta ph:(0,2*pi):\varphi")
XY

Chart (M, (t, r, th, ph))

# I. Definition of the metric

In [8]:
m, xi, a, B = var('m xi alpha B')
assume(m >= 0, 'real')
assume(xi > 0, 'real')
alpha = 1/(2*sqrt(3))

In [9]:
Omega = function('Omega')
Lambda = function('Lambda')
Av = function('A_v')
Ev = function('E_v')

In [10]:
g = M.metric()

g[0,0] = - Av(r) * Lambda(r,th)**(2/(1+alpha**2))*Omega(r,th)**2
g[1,1] = Ev(r,th) / Av(r) * Lambda(r,th)**(2/(1+alpha**2))*Omega(r,th)**2
g[2,2] = r**2 * Ev(r,th) * Lambda(r,th)**(2/(1+alpha**2))*Omega(r,th)**2
g[3,3] =  (r*sin(th))**2 / Lambda(r,th)**(2/(1+alpha**2))*Omega(r,th)**2

show(g.display())

g = -A_v(r)*Lambda(r, th)^(24/13)*Omega(r, th)^2 dt⊗dt + E_v(r, th)*Lambda(r, th)^(24/13)*Omega(r, th)^2/A_v(r) dr⊗dr + r^2*E_v(r, th)*Lambda(r, th)^(24/13)*Omega(r, th)^2 dth⊗dth + r^2*Omega(r, th)^2*sin(th)^2/Lambda(r, th)^(24/13) dph⊗dph

In [11]:
nab = g.connection(name=r'\nabla') 

# II. The vector potential

In [12]:
pot_vec = M.tensor_field(0,1,name='A')
pot_vec[0]=0
pot_vec[1]=0
pot_vec[2]=0
pot_vec[3]= - 2 / (1+alpha**2) / B / Lambda(r,th)

show(pot_vec.display())

A = -24/13/(B*Lambda(r, th)) dph

In [13]:
DF = nab(pot_vec) ; DF

Tensor field \nabla(A) of type (0,2) on the 4-dimensional Lorentzian manifold M

# III. Definition of the EM tensor

In [14]:
F = diff(pot_vec)
F.set_name('F')
Fuu = F.up(g)
F.display()

F = 24/13*d(Lambda)/dr/(B*Lambda(r, th)^2) dr∧dph + 24/13*d(Lambda)/dth/(B*Lambda(r, th)^2) dth∧dph

# IV. Variables and functions

In [15]:
Av_expr(r) = (1- 2*m /r)
Ev_expr(r,th) = exp(- (xi**2 * m**2 * r**2 * Av_expr(r) * sin(th)**2) / (2 * (r**2*Av_expr(r) + m**2 * cos(th)**2)**2))
old_Phi = M.scalar_field({XY:m*xi / sqrt(2*(r**2*Av_expr(r) + m**2 * cos(th)**2))}, name=r'\varphi')
old_Psi = exp(2*alpha*old_Phi)
Lambda_expr(r, th) = 1 + (1+ alpha**2) / 4 * B**2 * (r*sin(th))**2 * old_Psi.expr()

In [16]:
show(LatexExpr(r'\Lambda = '), Lambda_expr(r, th))
show(LatexExpr(r'A_v(r) = '), Av_expr(r), LatexExpr(r', \quad\quad    E_v(r,\theta) = '), Ev_expr(r,th))
show(LatexExpr(r"\phi_{0} = "), old_Phi.expr())
show(LatexExpr(r"\Psi_{0} = e^{2\alpha\phi_{0}} = "), old_Psi.expr())

\Lambda =  13/48*B^2*r^2*e^(1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2))*sin(th)^2 + 1

A_v(r) =  -2*m/r + 1 , \quad\quad    E_v(r,\theta) =  e^(1/2*m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2)

\phi_{0} =  m*xi/sqrt(2*m^2*cos(th)^2 - 2*r^2*(2*m/r - 1))

\Psi_{0} = e^{2\alpha\phi_{0}} =  e^(1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2))

In [17]:
Phi = M.scalar_field({XY: - alpha / (1+alpha**2) * log(Lambda(r,th)) + old_Phi.expr()}, name='Phi', latex_name=r'\Phi')
Psi = M.scalar_field({XY: old_Psi.expr()/ (Lambda(r,th))**(2*alpha**2/(1+alpha**2))}, name=r'\Psi')
varth = M.scalar_field({XY:Psi.expr()**(-1)}, name=r'\vartheta') # varth=1/Psi=e^{-2*\alpha*\phi}

Omega_expr(r,th) = Psi.expr()

show(LatexExpr(r"\Psi' = e^{2\alpha\phi'} = "), Psi.expr())
show(LatexExpr(r"\varphi' = "), Phi.expr())
show(LatexExpr(r"\Omega = e^{2\alpha\phi'} = \Psi' = "), Omega_expr(r,th))
show(LatexExpr(r"\vartheta = e^{-2\alpha\phi'} = \frac{1}{\Psi'} ="), varth.expr())

\Psi' = e^{2\alpha\phi'} =  e^(1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2))/Lambda(r, th)^(2/13)

\varphi' =  m*xi/sqrt(2*m^2*cos(th)^2 - 2*r^2*(2*m/r - 1)) - 2/13*sqrt(3)*log(Lambda(r, th))

\Omega = e^{2\alpha\phi'} = \Psi' =  e^(1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2))/Lambda(r, th)^(2/13)

\vartheta = e^{-2\alpha\phi'} = \frac{1}{\Psi'} = Lambda(r, th)^(2/13)*e^(-1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2))

## IV. 1. Metric and potential vector expressions

In [18]:
A_expr = - Av(r) * Lambda(r,th)**(2/(1+alpha**2))
B_expr = Ev(r,th) / Av(r) * Lambda(r,th)**(2/(1+alpha**2))
C_expr = r**2 * Ev(r,th) * Lambda(r,th)**(2/(1+alpha**2))
D_expr =  (r*sin(th))**2 / Lambda(r,th)**(2/(1+alpha**2))

In [19]:
g_nod = g.copy()
g_nod = subs_func(g_nod)
nod_ratio = (g_nod[3,3]/(sin(th)**2*g_nod[2,2])).factor()
show(LatexExpr(r"\lim_{\theta\rightarrow 0} \frac{g_{\phi\phi}}{\sin^{2}\theta g_{\theta\theta}} = " + latex(limit(nod_ratio.expr(), th=0).canonicalize_radical())))

Omega substituted
Lambda substituted
Av substituted
Ev substituted
Full Substitution : Done


\lim_{\theta\rightarrow 0} \frac{g_{\phi\phi}}{\sin^{2}\theta g_{\theta\theta}} = 1

# V. Field equations

## V. 1. The scalar-field form $-\frac{\mathcal{L}_m}{R} = \vartheta$

In [20]:
Lm = -F['_ij']*F.up(g)['^ij']/2

In [21]:
R = g.ricci_scalar() #Ricci scalar

In [22]:
Ric = g.ricci() #Ricci tensor

In [23]:
G = Ric-R*g/2 #Einstein tensor

In [24]:
eq = Lm/R + varth
eq = subs_func(eq)

Omega substituted
Lambda substituted
Av substituted
Ev substituted
Full Substitution : Done


In [25]:
var_dict = {"eq": eq}
save(var_dict, "ER_Penney_Melvin_dict.sobj")

In [59]:
latex_str = r'\frac{\mathcal{L}_m}{R} + \vartheta = ' 
show(LatexExpr(latex_str), numerator(eq.factor()).simplify_full())

\frac{\mathcal{L}_m}{R} + \vartheta =  0

## V. 2. Electromagnetic field equation

In [27]:
eq_MxW = nab(varth * Fuu)['_i^ij']
eq_MxW = subs_func(eq_MxW)
eq_MxW.apply_map(lambda f: f.simplify_trig())

Omega substituted
Lambda substituted
Av substituted
Ev substituted
Full Substitution : Done


In [28]:
latex_str = r'\triangledown_{\sigma}\left(\frac{\mathcal{L}_m}{R}F^{\mu\sigma}\right) = ' 
show(LatexExpr(latex_str), eq_MxW[:])

\triangledown_{\sigma}\left(\frac{\mathcal{L}_m}{R}F^{\mu\sigma}\right) =  [0, 0, 0, 0]

## V. 3. The metric field equation

Let's define the stress-energy tensor $T_{\mu\nu}=2\left(F_{\rho\mu}F^{\rho}_{\hspace{0.2cm}\nu}-\frac{1}{4}g_{\mu\nu}F^{2}\right)$ 

In [29]:
T = 2*(F.up(g,1)['_i^j']*F['_kj'] + g*Lm/2)

Let's now verify that the trace of the stress-energy tensor is zero

In [30]:
T.up(g,1).trace(0,1).display()

M → ℝ
(t, r, th, ph) ↦ 0

In [31]:
S = (nab(nab(varth**2)) - g*(varth**2).dalembertian()) / varth**2

And the field equation

In [32]:
eq_m = G + R / Lm * T - S

All the components $eq_m[i,j]$ of this tensor equation cannot be easily simplified to assess whether they vanish. Their explicit expressions involve long combinations of algebraic terms and exponential functions, whose direct simplification in SageMath is impractical due to their complexity. To demonstrate that these components are identically zero, we show that the numerator of $eq_m[i,j]$ can be written as:

$eq_m[i,j] = A + \sum_n C_n\,\exp(f_n)$

where $A$ and $C_n$ are algebraic expression and the set $\{ f_n\}_n$ contains no redundant terms. Since exponential functions never vanish, the algebraic contribution $A$ must be identically zero for the full expression to vanish.

### Component $eq_{00}$

In [33]:
test_00 = numerator(subs_func(eq_m[0,0]))

Omega substituted
Lambda substituted
Av substituted
Ev substituted
Full Substitution : Done


In [34]:
test_00.simplify_full()

0

### Component $eq_{33}$

In [35]:
test_33 = numerator(subs_func(eq_m[3,3]))

Omega substituted
Lambda substituted
Av substituted
Ev substituted
Full Substitution : Done


In [36]:
test_33.simplify_full()

0

### Component $eq_{11}$

In [37]:
test_11 = eq_m[1,1].expr().substitute_function(Omega, Omega_expr).substitute_function(Lambda,Lambda_expr)\
                                .substitute_function(Av,Av_expr).substitute_function(Ev,Ev_expr)


In [38]:
'''The expression'''
expr = test_11
show(LatexExpr(r"\text{Expression's length :}"),len(repr(expr)))

'''Clean the expression'''
show(LatexExpr(r' \text{Are there any exponentials with zero arguments? :}'), explore_expression().found_zero)
clean_expr = explore_expression()(expr) # The expression is now free of exponentials with zero arguments

'''Check the list of exponential arguments'''
exp_list = explore_expression()
_ = exp_list(expr)
uni_list = list(set(exp_list.exp_args))
show(LatexExpr(r' \text{The arguments in the exponentials are :}'), uni_list)

\text{Expression's length :} 209209

\text{Are there any exponentials with zero arguments? :} False

\text{The arguments in the exponentials are :} [m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2),
 1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2),
 m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2,
 1/2*m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2,
 m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/3*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2),
 1/2*m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/3*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2)]

In [39]:
'''Check the algebraic part of the expression'''
res = check_res_expr()
show(LatexExpr(r' \text{Algebraic part is zero: } '),res(numerator(clean_expr)).is_zero())

\text{Algebraic part is zero: }  True

In [40]:
'''Isolate each exponential individually and check its multiplying factor'''
for n, arg in enumerate(uni_list):
    dec = decomp_exp(arg)
    isol_expr = dec(expr)
    show(LatexExpr(r'\text{The factor associated with: }'),\
         arg, LatexExpr(r' \text{ is zero: }'), isol_expr.is_zero())

\text{The factor associated with: } m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2)  \text{ is zero: } True

\text{The factor associated with: } 1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2)  \text{ is zero: } True

\text{The factor associated with: } m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2  \text{ is zero: } True

\text{The factor associated with: } 1/2*m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2  \text{ is zero: } True

\text{The factor associated with: } m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/3*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2)  \text{ is zero: } True

\text{The factor associated with: } 1/2*m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/3*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2)  \text{ is zero: } True

### Component $eq_{22}$

In [41]:
test_22 = eq_m[2,2].expr().substitute_function(Omega, Omega_expr).substitute_function(Lambda,Lambda_expr)\
                                .substitute_function(Av,Av_expr).substitute_function(Ev,Ev_expr)


In [42]:
'''The expression'''
expr = test_22
show(LatexExpr(r"\text{Expression's length :}"),len(repr(expr)))

'''Clean the expression'''
show(LatexExpr(r' \text{Are there any exponentials with zero arguments? :}'), explore_expression().found_zero)
clean_expr = explore_expression()(expr) # The expression is now free of exponentials with zero arguments

'''Check the list of exponential arguments'''
exp_list = explore_expression()
_ = exp_list(expr)
uni_list = list(set(exp_list.exp_args))
show(LatexExpr(r' \text{The arguments in the exponentials are :}'), uni_list)

\text{Expression's length :} 209014

\text{Are there any exponentials with zero arguments? :} False

\text{The arguments in the exponentials are :} [m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2),
 1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2),
 m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2,
 1/2*m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2,
 m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/3*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2),
 1/2*m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/3*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2)]

In [43]:
'''Check the algebraic part of the expression'''
res = check_res_expr()
show(LatexExpr(r' \text{Algebraic part is zero: } '),res(numerator(clean_expr)).is_zero())

\text{Algebraic part is zero: }  True

In [44]:
'''Isolate each exponential individually and check its multiplying factor'''
for n, arg in enumerate(uni_list):
    dec = decomp_exp(arg)
    isol_expr = dec(expr)
    show(LatexExpr(r'\text{The factor associated with: }'),\
         arg, LatexExpr(r' \text{ is zero: }'), isol_expr.is_zero())

\text{The factor associated with: } m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2)  \text{ is zero: } True

\text{The factor associated with: } 1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2)  \text{ is zero: } True

\text{The factor associated with: } m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2  \text{ is zero: } True

\text{The factor associated with: } 1/2*m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2  \text{ is zero: } True

\text{The factor associated with: } m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/3*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2)  \text{ is zero: } True

\text{The factor associated with: } 1/2*m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/3*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2)  \text{ is zero: } True

### Component $eq_{12}$

In [45]:
test_12 = eq_m[1,2].expr().substitute_function(Omega, Omega_expr).substitute_function(Lambda,Lambda_expr)\
                                .substitute_function(Av,Av_expr).substitute_function(Ev,Ev_expr)


In [46]:
'''The expression'''
expr = test_12
show(LatexExpr(r"\text{Expression's length :}"),len(repr(expr)))

'''Clean the expression'''
show(LatexExpr(r' \text{Are there any exponentials with zero arguments? :}'), explore_expression().found_zero)
clean_expr = explore_expression()(expr) # The expression is now free of exponentials with zero arguments

'''Check the list of exponential arguments'''
exp_list = explore_expression()
_ = exp_list(expr)
uni_list = list(set(exp_list.exp_args))
show(LatexExpr(r' \text{The arguments in the exponentials are :}'), uni_list)

\text{Expression's length :} 201470

\text{Are there any exponentials with zero arguments? :} False

\text{The arguments in the exponentials are :} [m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2),
 1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2),
 m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2,
 1/2*m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2,
 m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/3*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2),
 1/2*m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/3*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2)]

In [47]:
'''Check the algebraic part of the expression'''
res = check_res_expr()
show(LatexExpr(r' \text{Algebraic part is zero: } '),res(numerator(clean_expr)).is_zero())

\text{Algebraic part is zero: }  True

In [48]:
'''Isolate each exponential individually and check its multiplying factor'''
for n, arg in enumerate(uni_list):
    dec = decomp_exp(arg)
    isol_expr = dec(expr)
    show(LatexExpr(r'\text{The factor associated with: }'),\
         arg, LatexExpr(r' \text{ is zero: }'), isol_expr.is_zero())

\text{The factor associated with: } m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2)  \text{ is zero: } True

\text{The factor associated with: } 1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2)  \text{ is zero: } True

\text{The factor associated with: } m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2  \text{ is zero: } True

\text{The factor associated with: } 1/2*m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2  \text{ is zero: } True

\text{The factor associated with: } m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/3*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2)  \text{ is zero: } True

\text{The factor associated with: } 1/2*m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/3*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2)  \text{ is zero: } True

### Component $eq_{21}$

In [49]:
test_21 = eq_m[2,1].expr().substitute_function(Omega, Omega_expr).substitute_function(Lambda,Lambda_expr)\
                                .substitute_function(Av,Av_expr).substitute_function(Ev,Ev_expr)


In [50]:
'''The expression'''
expr = test_21
show(LatexExpr(r"\text{Expression's length :}"),len(repr(expr)))

'''Clean the expression'''
show(LatexExpr(r' \text{Are there any exponentials with zero arguments? :}'), explore_expression().found_zero)
clean_expr = explore_expression()(expr) # The expression is now free of exponentials with zero arguments

'''Check the list of exponential arguments'''
exp_list = explore_expression()
_ = exp_list(expr)
uni_list = list(set(exp_list.exp_args))
show(LatexExpr(r' \text{The arguments in the exponentials are :}'), uni_list)

\text{Expression's length :} 201470

\text{Are there any exponentials with zero arguments? :} False

\text{The arguments in the exponentials are :} [m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2),
 1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2),
 m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2,
 1/2*m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2,
 m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/3*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2),
 1/2*m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/3*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2)]

In [51]:
'''Check the algebraic part of the expression'''
res = check_res_expr()
show(LatexExpr(r' \text{Algebraic part is zero: } '),res(numerator(clean_expr)).is_zero())

\text{Algebraic part is zero: }  True

In [52]:
'''Isolate each exponential individually and check its multiplying factor'''
for n, arg in enumerate(uni_list):
    dec = decomp_exp(arg)
    isol_expr = dec(expr)
    show(LatexExpr(r'\text{The factor associated with: }'),\
         arg, LatexExpr(r' \text{ is zero: }'), isol_expr.is_zero())

\text{The factor associated with: } m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2)  \text{ is zero: } True

\text{The factor associated with: } 1/6*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2)  \text{ is zero: } True

\text{The factor associated with: } m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2  \text{ is zero: } True

\text{The factor associated with: } 1/2*m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2  \text{ is zero: } True

\text{The factor associated with: } m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/3*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2)  \text{ is zero: } True

\text{The factor associated with: } 1/2*m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2 + 1/3*sqrt(3)*sqrt(2)*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2)  \text{ is zero: } True

In [53]:
var_dict = {"test_00": test_00,"test_33": test_33,"test_11": test_11, "test_22": test_22, "test_12": test_12, "test_21": test_21, "eq_m": eq_m}
save(var_dict, "ER_Penney_Melvin_test_dict.sobj")

In [54]:
eq_result = eq_m.copy()

for i in range(4):
    for j in range(4):
        if [i,j] in [[0,0],[1,1],[2,2],[3,3],[1,2],[2,1]]:
            eq_result[i,j]=0

In [55]:
latex_str = r'G_{\mu\nu} + \frac{R}{\mathcal{L}_{m}}T_{\mu\nu}-\frac{1}{\vartheta^2}\left[\nabla_{\mu}\nabla_{\nu} - g_{\mu\nu}\square\right]\vartheta^{2} = ' 
show(LatexExpr(latex_str), eq_result[:])

G_{\mu\nu} + \frac{R}{\mathcal{L}_{m}}T_{\mu\nu}-\frac{1}{\vartheta^2}\left[\nabla_{\mu}\nabla_{\nu} - g_{\mu\nu}\square\right]\vartheta^{2} =  [0 0 0 0]
[0 0 0 0]
[0 0 0 0]
[0 0 0 0]

# WE CONCLUDE THAT EMBEDDING THE PENNEY SOLUTION INTO AN EXTERNAL MAGNETIC FIELD GENERATE A NEW SOLUTION IN ER